# Mask2Former Training — Lane Detection

**Architecture:** Mask2Former + Swin-Small backbone  
**Dataset:** Lane Detection (5 classes, 1610 images)  
**Strategy:** Gradual freezing (3 phases) + CosineAnnealingWarmRestarts  
**Tracking:** MLflow → DagsHub  

**Open this notebook from GitHub** (not an old copy on Drive):  
https://colab.research.google.com/github/srnortw/mask2former/blob/main/notebooks/train_colab.ipynb

**Session layout:**
- **Code** → `git clone` / `pull` to `/content/mask2former` (fast Colab disk; wiped when runtime ends)
- **Artifacts on Drive** → `drive.mount` + `mask2former-mlops/data/` + `checkpoints/` (persist)

```
mask2former-mlops/          ← Google Drive only
├── data/raw/               ← Roboflow dataset (Cell 5)
└── checkpoints/            ← .pth, .onnx

/content/mask2former/       ← git clone from GitHub (Cells 4–5)
```

- **Cell 4:** `drive.mount` + set `CHECKPOINT_DIR` / `DATA_DIR`
- **Cell 5:** `git clone` or `pull` → `/content/mask2former` + Roboflow if needed

You can delete the old `mask2former-mlops/mask2former/` folder on Drive (no longer used).

---
**Before running:** Runtime → GPU (T4). Colab secrets: `ROBOFLOW_API_KEY`, `HF_TOKEN`, MLflow (repo is public — no `GITHUB_TOKEN` needed for clone).

In [ ]:
# ── Cell 1: Verify GPU ──────────────────────────────────────────────────────
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU:  {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'torch: {torch.__version__}')

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
!pip install -q transformers accelerate
!pip install -q mlflow pymongo
!pip install -q albumentations pycocotools
!pip install -q roboflow dvc python-dotenv
print('Dependencies installed')

In [ ]:
# ── Cell 3: Load secrets from Colab ─────────────────────────────────────────
# Add these in Colab: left sidebar → 🔑 Secrets
#   ROBOFLOW_API_KEY, HF_TOKEN, MONGO_URI, MLFLOW_TRACKING_URI
import os
from google.colab import userdata

os.environ['ROBOFLOW_API_KEY']    = userdata.get('ROBOFLOW_API_KEY')
os.environ['ROBOFLOW_WORKSPACE']  = 'test-mfeql'
os.environ['ROBOFLOW_PROJECT']    = 'lane-detection-segmentation-edyqp-fibkz'
os.environ['ROBOFLOW_VERSION']    = '1'
os.environ['HF_TOKEN']            = userdata.get('HF_TOKEN')
os.environ['HF_REPO_ID']          = 'srnortw/mask2former-lane-seg'
os.environ['MONGO_URI']           = userdata.get('MONGO_URI')
os.environ['MONGO_DB_NAME']       = 'mask2former'
os.environ['MONGO_COLLECTION_PREDICTIONS'] = 'predictions'
os.environ['MONGO_COLLECTION_DRIFT']       = 'drift_reports'
os.environ['MLFLOW_TRACKING_URI']      = userdata.get('MLFLOW_TRACKING_URI')
os.environ['MLFLOW_TRACKING_USERNAME']  = 'srnortw'
os.environ['MLFLOW_TRACKING_PASSWORD']  = userdata.get('MLFLOW_TRACKING_PASSWORD')
os.environ['MLFLOW_EXPERIMENT_NAME']    = 'mask2former-swin'
# Optional — only if you git push from Colab (clone works without token; repo is public)
try:
    os.environ['GITHUB_TOKEN'] = userdata.get('GITHUB_TOKEN')
except userdata.SecretNotFoundError:
    pass
os.environ['GITHUB_REPO'] = 'srnortw/mask2former'
print('Secrets loaded')

In [ ]:
# ── Cell 4: Mount Drive (artifacts only) ─────────────────────────────────────
# Code lives on GitHub → /content/mask2former (Cell 5). Drive = data + checkpoints only.
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=False)

MYDRIVE = '/content/drive/MyDrive'
DRIVE_MLOPS = f'{MYDRIVE}/mask2former-mlops'

assert os.path.isdir(MYDRIVE), 'Drive mount failed — approve access in the popup.'

os.environ['DRIVE_MLOPS'] = DRIVE_MLOPS
os.environ['CHECKPOINT_DIR'] = f'{DRIVE_MLOPS}/checkpoints'
os.environ['DATA_DIR'] = f'{DRIVE_MLOPS}/data'

for d in (DRIVE_MLOPS, os.environ['CHECKPOINT_DIR'], os.environ['DATA_DIR']):
    os.makedirs(d, exist_ok=True)

print(f'✓ Drive mounted')
print(f'  Checkpoints: {os.environ["CHECKPOINT_DIR"]}')
print(f'  Data:        {os.environ["DATA_DIR"]}/raw')
print('  Next: run Cell 5 (git clone → /content/mask2former)')

In [ ]:
# ── Cell 5: Clone code from GitHub + Roboflow data ───────────────────────────
# Run Cell 4 first (Drive mount). Code → /content/mask2former (not on Drive).
import os, sys, shutil, subprocess

assert os.environ.get('CHECKPOINT_DIR'), 'Run Cell 4 first (Drive mount).'

REPO = '/content/mask2former'
GIT_URL = 'https://github.com/srnortw/mask2former.git'
os.environ['PROJECT_ROOT'] = REPO

git_dir = os.path.join(REPO, '.git')
if os.path.isdir(git_dir):
    subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', GIT_URL], check=False)
    subprocess.run(['git', '-C', REPO, 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', REPO, 'reset', '--hard', 'origin/main'], check=True)
    print(f'✓ Code synced: {REPO}')
else:
    if os.path.isdir(REPO):
        shutil.rmtree(REPO)
    print(f'git clone → {REPO}')
    subprocess.run(['git', 'clone', GIT_URL, REPO], check=True)
    subprocess.run(['git', '-C', REPO, 'reset', '--hard', 'origin/main'], check=True)

os.chdir(REPO)
sys.path.insert(0, f'{REPO}/src')
print(f'✓ Project:     {REPO}')
print(f'  Checkpoints: {os.environ["CHECKPOINT_DIR"]}')
print(f'  Data:        {os.environ["DATA_DIR"]}/raw')
print(f'  cwd:         {os.getcwd()}')

raw_dir = os.path.join(os.environ['DATA_DIR'], 'raw')
ann = os.path.join(raw_dir, 'train/_annotations.coco.json')
if os.path.isfile(ann):
    print('data/raw on Drive — skipping Roboflow download')
else:
    from roboflow import Roboflow
    rf = Roboflow(api_key=os.environ['ROBOFLOW_API_KEY'])
    project = rf.workspace('test-mfeql').project('lane-detection-segmentation-edyqp-fibkz')
    dataset = project.version(1).download('coco-segmentation', location=raw_dir, overwrite=False)
    print(f'Data ready at: {dataset.location}')

In [ ]:
# ── Cell 6: Load config and verify data ──────────────────────────────────────
import sys, os
ROOT = os.environ['PROJECT_ROOT']
sys.path.insert(0, f'{ROOT}/src')
os.chdir(ROOT)

from config_loader import load_config
from data.dataset import build_dataloaders

# Colab: all artifacts on Drive (not inside the git repo)
cfg = load_config()
CKPT = os.environ['CHECKPOINT_DIR']
DATA = os.environ['DATA_DIR']

cfg.model.checkpoint_dir = CKPT + '/'
raw_dir = os.path.join(DATA, 'raw')
legacy_raw = os.path.join(ROOT, 'data/raw')
if not os.path.isfile(os.path.join(raw_dir, 'train/_annotations.coco.json')) and os.path.isfile(
    os.path.join(legacy_raw, 'train/_annotations.coco.json')
):
    raw_dir = legacy_raw
    print('NOTE: using data/raw inside clone — prefer Drive data/raw (Cell 5 Roboflow)')
cfg.data.raw_dir = raw_dir
cfg.data.processed_dir = os.path.join(DATA, 'processed')
cfg.data.calibration_dir = os.path.join(DATA, 'calibration')
cfg.onnx.output.fp32 = os.path.join(CKPT, 'mask2former_fp32.onnx')
cfg.onnx.output.int8 = os.path.join(CKPT, 'mask2former_int8.onnx')

os.makedirs(CKPT, exist_ok=True)
print(f'Checkpoints → {cfg.model.checkpoint_dir}')
print(f'Data        → {cfg.data.raw_dir}')

train_loader, val_loader = build_dataloaders(cfg)
print(f'Train: {len(train_loader.dataset)} samples')
print(f'Val:   {len(val_loader.dataset)} samples')

# Quick batch check
import torch
imgs, targets = next(iter(train_loader))
print(f'Batch shape: {imgs.shape}')
print(f'Sample masks: {targets[0]["masks"].shape}')

In [ ]:
# ── Cell 7: Build model + verify freezing ────────────────────────────────────
import sys, os
sys.path.insert(0, f"{os.environ['PROJECT_ROOT']}/src")
from models.mask2former import build_model, set_phase

device = torch.device('cuda')
model = build_model(cfg).to(device)

# Show phase 1 trainable params
set_phase(model, phase=1)

# Sanity forward pass
model.eval()
with torch.no_grad():
    out = model(pixel_values=imgs.to(device))
print(f'Output keys: {list(out.keys())}')
print(f'Masks shape: {out.masks_queries_logits.shape}')
print(f'Logits shape: {out.class_queries_logits.shape}')

In [ ]:
# ── Cell 8: Check MLflow connection ─────────────────────────────────────────
import mlflow
mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])
mlflow.set_experiment(os.environ['MLFLOW_EXPERIMENT_NAME'])

# Test connection with a quick log
with mlflow.start_run(run_name='connection-test') as run:
    mlflow.log_param('test', True)
    mlflow.log_metric('test_metric', 1.0)
    print(f'MLflow run ID: {run.info.run_id}')
    print(f'Tracking URI: {mlflow.get_tracking_uri()}')
print('MLflow connection OK')

In [ ]:
# ── Cell 9: Phase 1 — Transformer decoder only (epochs 0–15) ────────────────
# Backbone + pixel decoder frozen. Only the transformer decoder + class head train.
# Expected: loss drops fast (~48→25), mAP still 0 (scores below threshold — normal).
import sys, os
sys.path.insert(0, f"{os.environ['PROJECT_ROOT']}/src")
from train import train

best_map, run_id = train(cfg, stop_at_epoch=15)
print(f'\nPhase 1 done. best_mAP={best_map:.4f}  run_id={run_id}')

In [ ]:
# ── Cell 10: Phase 2 — Pixel decoder unfreezes (epochs 15–30) ────────────────
# Backbone still frozen. Feature pyramid now adapts to your lane data.
# Expected: loss continues dropping (~25→15), mAP may start appearing.
# Auto-resumes from last_checkpoint.pth and continues same MLflow run.

best_map, run_id = train(cfg, stop_at_epoch=30)
print(f'\nPhase 2 done. best_mAP={best_map:.4f}  run_id={run_id}')

In [ ]:
# ── Cell 11: Phase 3 — Full fine-tune (epochs 30–50) ────────────────────────
# All layers unfreeze with low LR. This is where real mAP gains happen.
# Expected: loss stabilizes (~10–15), mAP rises above 0.2+.
# Auto-resumes from last_checkpoint.pth and continues same MLflow run.

best_map, run_id = train(cfg, stop_at_epoch=50)
print(f'\nTraining complete! best_mAP={best_map:.4f}  run_id={run_id}')

In [ ]:
# ── Cell 12: Verify checkpoints on Google Drive ─────────────────────────────
# Training writes directly to CHECKPOINT_DIR (set in Cell 4/5). No copy step needed.
import os

CKPT = os.environ['CHECKPOINT_DIR']
for ckpt in ['best_model.pth', 'phase1_final.pth', 'phase2_final.pth', 'phase3_final.pth', 'last_checkpoint.pth']:
    path = os.path.join(CKPT, ckpt)
    if os.path.isfile(path):
        mb = os.path.getsize(path) / 1e6
        print(f'  {ckpt}: {mb:.1f} MB')
    else:
        print(f'  {ckpt}: (not found)')
print(f'\nDrive folder: {CKPT}')

In [ ]:
# ── Cell 13: Push checkpoint to Hugging Face Hub ─────────────────────────────
import os
from huggingface_hub import HfApi, create_repo

api = HfApi(token=os.environ['HF_TOKEN'])
repo_id = os.environ['HF_REPO_ID']

create_repo(repo_id, repo_type='model', private=True, exist_ok=True)

best_ckpt = os.path.join(os.environ['CHECKPOINT_DIR'], 'best_model.pth')
api.upload_file(
    path_or_fileobj=best_ckpt,
    path_in_repo='best_model.pth',
    repo_id=repo_id,
)
print(f'Model pushed to: https://huggingface.co/{repo_id}')

---
## Phase 2 — ONNX Export + INT8 Quantization

Run these cells after training (or download `best_model.pth` from HF in Cell 16).  
**Requires:** `best_model.pth` in Drive `mask2former-mlops/checkpoints/`

In [ ]:
# ── Cell 14: Install ONNX dependencies ──────────────────────────────────────
!pip install -q onnx onnxruntime onnxruntime-tools onnxscript
print('ONNX dependencies installed')

In [ ]:
# ── Cell 15: Download checkpoint from HuggingFace Hub ───────────────────────
# Run after Colab restart if best_model.pth is not already on Drive.
from huggingface_hub import hf_hub_download
import os

CKPT = os.environ['CHECKPOINT_DIR']
os.makedirs(CKPT, exist_ok=True)
best = os.path.join(CKPT, 'best_model.pth')

if os.path.isfile(best):
    print(f'Already on Drive: {best}')
else:
    hf_hub_download(
        repo_id=os.environ['HF_REPO_ID'],
        filename='best_model.pth',
        local_dir=CKPT,
        token=os.environ['HF_TOKEN'],
    )
    print(f'Downloaded: {best}')

In [ ]:
# ── Cell 16: Export PyTorch → ONNX fp32 (opset 16) ───────────────────────────
# Opset 16 uses the TorchScript exporter (dynamo exporter only at opset 18+).
# TorchScript exporter is required for static INT8 quantization compatibility.
import sys, os
ROOT = os.environ['PROJECT_ROOT']
sys.path.insert(0, f'{ROOT}/src')
os.chdir(ROOT)

from export_onnx import export_to_onnx, verify_onnx

CKPT = os.environ['CHECKPOINT_DIR']
ckpt_path = os.path.join(CKPT, 'best_model.pth')
fp32_path = os.path.join(CKPT, 'mask2former_fp32.onnx')

export_to_onnx(
    checkpoint_path=ckpt_path,
    output_path=fp32_path,
    img_size=512,
    opset_version=16,   # TorchScript exporter — static quantization compatible
    device='cpu',
)

verify_onnx(ckpt_path, fp32_path, img_size=512)

---
## Phase 4b — FiftyOne evaluation (valid + test)

After export (Cell 16) and INT8 (Cell 17), compare **fp32 ONNX vs INT8 ONNX** on **valid** and **test** with COCO mask mAP + FiftyOne App (PR / AP plots).

**Phase 3 `evaluate.py`** already scores **PyTorch** on val during training — keep that for checkpoint picking. This cell validates **exported artifacts** only.

| Model | When | Why |
|-------|------|-----|
| PyTorch | End of training (Cell 9–11) | Best checkpoint / mAP for training |
| **fp32 ONNX** | After Cell 16 | Export correctness |
| **int8 ONNX** | After Cell 17 | What you deploy (HF + Docker) |

Optional: `max_samples=50` for a quick Colab pass; `None` for full valid/test.

In [ ]:
# ── Cell 19: FiftyOne eval — valid + test (fp32 vs INT8 ONNX) ───────────────
!pip install -q fiftyone

import sys, os
ROOT = os.environ['PROJECT_ROOT']
sys.path.insert(0, ROOT)
os.chdir(ROOT)

from src.evaluate_onnx_fiftyone import run_phase4_evaluation

CKPT = os.environ['CHECKPOINT_DIR']
fp32_path = os.path.join(CKPT, 'mask2former_fp32.onnx')
int8_path = os.path.join(CKPT, 'mask2former_int8.onnx')

# Data on Drive (Cell 4/5) — not inside /content/mask2former clone
assert os.environ.get('DATA_DIR'), 'Run Cells 4–5 first (Drive mount + data paths).'
RAW = os.path.join(os.environ['DATA_DIR'], 'raw')
assert os.path.isdir(os.path.join(RAW, 'valid')), (
    f'Missing {RAW}/valid — run Cell 5 (Roboflow) or copy data to Drive data/raw'
)
print(f'FiftyOne raw_dir: {RAW}')

summary = run_phase4_evaluation(
    fp32_onnx_path=fp32_path,
    int8_onnx_path=int8_path,
    splits=['valid', 'test'],
    raw_dir=RAW,
    img_size=512,
    score_threshold=0.5,
    max_samples=None,   # set e.g. 50 for faster Colab run
    launch_app=True,
)

print('\n=== mAP summary ===')
for split, models in summary.items():
    for tag, info in models.items():
        print(f"  {split} / {tag}: mAP = {info.get('mAP')}")

In [ ]:
# ── Cell 17: Selective Static INT8 Quantization ──────────────────────────────
# Quantizes Conv/MatMul/Gemm only (Swin backbone heavy compute).
# Deformable attention stays fp32 (ONNX shape inference incompatibility).
#
# Type pairing (Q-ViT / FQ-ViT recommendation):
#   weight_type=QInt8   — weights can be negative
#   activation_type=QUInt8 — post-Softmax/GELU in [0,1], unsigned
#
# Calibrated on val set: domain-specific, not seen during training.
import os
from quantize_int8 import quantize_int8, benchmark, _get_calibration_dir

int8_path = os.path.join(os.environ['CHECKPOINT_DIR'], 'mask2former_int8.onnx')
cal_dir = _get_calibration_dir(cfg)

quantize_int8(
    fp32_onnx_path=fp32_path,
    int8_onnx_path=int8_path,
    calibration_dir=cal_dir,
    img_size=512,
    n_images=200,
)

print('\nBenchmark (CPU):')
t_fp32 = benchmark(fp32_path, img_size=512, n_runs=10)
t_int8 = benchmark(int8_path, img_size=512, n_runs=10)
print(f'Speedup: {t_fp32/t_int8:.1f}x')

In [ ]:
# ── Cell 20: Push ONNX models to HuggingFace Hub ─────────────────────────────
# ONNX files are already on Drive (Cells 16–17). Run Cell 19 (FiftyOne) before push.
from huggingface_hub import HfApi
import os

api = HfApi(token=os.environ['HF_TOKEN'])
repo_id = os.environ['HF_REPO_ID']
CKPT = os.environ['CHECKPOINT_DIR']

for fname in ['mask2former_fp32.onnx', 'mask2former_int8.onnx']:
    local = os.path.join(CKPT, fname)
    if os.path.isfile(local):
        api.upload_file(
            path_or_fileobj=local,
            path_in_repo=fname,
            repo_id=repo_id,
        )
        size_mb = os.path.getsize(local) / 1e6
        print(f'Pushed: {fname} ({size_mb:.1f} MB) → https://huggingface.co/{repo_id}')
        print(f'  Drive: {local}')
    else:
        print(f'Skip (missing): {local}')

print('\nONNX phase complete.')

---
## Phase 3 — Model Registry

Register the model in MLflow Model Registry (Staging) and push the model card to HF Hub.

In [ ]:
# ── Cell 20: Register model in MLflow + push model card to HF Hub ────────────
# Run Cells 4–5 first (Drive mount + git clone to /content/mask2former).
import sys, os, torch, subprocess, importlib
ROOT = os.environ['PROJECT_ROOT']
sys.path.insert(0, f'{ROOT}/src')
os.chdir(ROOT)

subprocess.run(['git', '-C', ROOT, 'fetch', 'origin', 'main'], check=True)
subprocess.run(['git', '-C', ROOT, 'reset', '--hard', 'origin/main'], check=True)
print('git sync OK (register_model.py up to date)')
import register_model as _reg
importlib.reload(_reg)
from register_model import register_in_mlflow, push_model_card, REGISTRY_CODE_VERSION
print(f'Loaded: {_reg.__file__}')
print(f'Expected banner: register_model.py [{REGISTRY_CODE_VERSION}]')

CKPT = os.environ['CHECKPOINT_DIR']

# Get run_id from the last checkpoint saved during training (on Drive)
last_ckpt_path = os.path.join(CKPT, 'last_checkpoint.pth')
last_ckpt = torch.load(last_ckpt_path, map_location='cpu')
run_id = last_ckpt.get('mlflow_run_id')

if run_id is None:
    # If last_checkpoint.pth isn't available, paste your run_id manually:
    run_id = 'PASTE_YOUR_RUN_ID_HERE'

print(f'Using MLflow run_id: {run_id}')

version = register_in_mlflow(
    run_id=run_id,
    checkpoint_path=os.path.join(CKPT, 'best_model.pth'),
    fp32_onnx_path=os.path.join(CKPT, 'mask2former_fp32.onnx'),
    int8_onnx_path=os.path.join(CKPT, 'mask2former_int8.onnx'),
)

# Push model card (README) to HF Hub
push_model_card(
    repo_id=os.environ['HF_REPO_ID'],
    hf_token=os.environ['HF_TOKEN'],
)

print(f'\nModel registry complete.')
print(f'Version {version} is in Staging.')
print(f'Run promote_to_production("{version}") after ROS2 validation.')